In [4]:
import pandas as pd

In [5]:
!git clone https://github.com/NicoLatina/proyecto-logistica.git

fatal: destination path 'proyecto-logistica' already exists and is not an empty directory.


In [6]:
df = pd.read_csv("/content/proyecto-logistica/data/processed/logistics_clean.csv", parse_dates=["last_restock_date"])

In [7]:
df.dtypes

,0
item_id,object
category,object
stock_level,int64
reorder_point,int64
reorder_frequency_days,int64
lead_time_days,int64
daily_demand,float64
demand_std_dev,float64
item_popularity_score,float64
storage_location_id,object


## STOCK DE COBERTURA

In [8]:
zeros_count = (df['daily_demand'] == 0).sum()
print(f'Cantidad de registros con daily_demand igual a 0: {zeros_count}')

if zeros_count > 0:
    display(df[df['daily_demand'] == 0].head())
else:
    print('No se encontraron valores en 0 en la columna daily_demand.')

Cantidad de registros con daily_demand igual a 0: 0
No se encontraron valores en 0 en la columna daily_demand.


In [9]:
df["days_of_stock_coverage"] = df["stock_level"] / df["daily_demand"]
display(df[["item_id", "stock_level", "daily_demand", "days_of_stock_coverage"]].head())

,item_id,stock_level,daily_demand,days_of_stock_coverage
0,ITM10000,283,49.85,5.677031
1,ITM10001,301,23.34,12.896315
2,ITM10002,132,37.69,3.502255
3,ITM10003,346,33.69,10.270110
4,ITM10004,49,49.58,0.988302


### Estadísticas Descriptivas de Cobertura de Stock
Calculamos los valores clave para entender la distribución de la cobertura en días.

In [10]:
stats = df['days_of_stock_coverage'].agg(['min', 'max', 'mean', 'median'])
print("Resumen Estadístico de Cobertura (Días):")
print(stats)

Resumen Estadístico de Cobertura (Días):
min         0.451773
max       422.772277
mean       21.004652
median     10.316800
Name: days_of_stock_coverage, dtype: float64


### Top 10 SKUs con MENOR Cobertura


In [11]:
low_coverage = df[['item_id', 'stock_level', 'daily_demand', 'days_of_stock_coverage']].nsmallest(10, 'days_of_stock_coverage')
display(low_coverage)

,item_id,stock_level,daily_demand,days_of_stock_coverage
11,ITM10011,20,44.27,0.451773
1458,ITM11458,22,46.53,0.472813
988,ITM10988,21,44.20,0.475113
2360,ITM12360,24,48.87,0.491099
2375,ITM12375,21,42.40,0.495283
966,ITM10966,25,48.18,0.518888
2457,ITM12457,25,47.59,0.525320
2728,ITM12728,27,49.74,0.542823
2039,ITM12039,25,44.81,0.557911
1554,ITM11554,21,37.25,0.563758


### Top 10 SKUs con MAYOR Cobertura

In [12]:
high_coverage = df[['item_id', 'stock_level', 'daily_demand', 'days_of_stock_coverage']].nlargest(10, 'days_of_stock_coverage')
display(high_coverage)

,item_id,stock_level,daily_demand,days_of_stock_coverage
2502,ITM12502,427,1.01,422.772277
2117,ITM12117,488,1.18,413.559322
694,ITM10694,460,1.13,407.079646
1886,ITM11886,426,1.07,398.130841
1230,ITM11230,463,1.17,395.726496
2459,ITM12459,489,1.28,382.031250
492,ITM10492,374,1.11,336.936937
340,ITM10340,440,1.32,333.333333
659,ITM10659,381,1.15,331.304348
3163,ITM13163,454,1.39,326.618705


## Cálculo de Demanda Durante Lead Time
Calculamos cuántas unidades se consumen durante el tiempo que tarda el proveedor en entregar un pedido.

In [13]:
df['expected_demand_during_lead_time'] = df['daily_demand'] * df['lead_time_days']
display(df[['item_id', 'daily_demand', 'lead_time_days', 'expected_demand_during_lead_time']].head())

,item_id,daily_demand,lead_time_days,expected_demand_during_lead_time
0,ITM10000,49.85,4,199.40
1,ITM10001,23.34,6,140.04
2,ITM10002,37.69,8,301.52
3,ITM10003,33.69,5,168.45
4,ITM10004,49.58,6,297.48


## Comparación de Stock vs Consumo Esperado (Gap)
Calculamos la diferencia entre el stock actual y lo que consumiremos mientras esperamos al proveedor.

In [14]:
df['lead_time_stock_gap'] = df['stock_level'] - df['expected_demand_during_lead_time']

# Mostramos los resultados principales para los primeros items
display(df[['item_id', 'stock_level', 'expected_demand_during_lead_time', 'lead_time_stock_gap']].head())

,item_id,stock_level,expected_demand_during_lead_time,lead_time_stock_gap
0,ITM10000,283,199.40,83.60
1,ITM10001,301,140.04,160.96
2,ITM10002,132,301.52,-169.52
3,ITM10003,346,168.45,177.55
4,ITM10004,49,297.48,-248.48


In [15]:
# Identificamos cuántos items están en riesgo de quiebre (gap < 0)
items_en_riesgo = df[df['lead_time_stock_gap'] < 0].shape[0]
print(f'Total de SKUs con riesgo de quiebre antes de la entrega: {items_en_riesgo}')

Total de SKUs con riesgo de quiebre antes de la entrega: 818


## Análisis de Punto de Reposición (Reorder Point)
Identificamos qué productos han alcanzado o superado su nivel crítico para realizar un nuevo pedido.

In [16]:
df['below_reorder_point'] = df['stock_level'] <= df['reorder_point']

# Calculamos totales y porcentajes
total_below = df['below_reorder_point'].sum()
perc_below = (total_below / len(df)) * 100

print(f'Total de SKUs debajo del punto de reposición: {total_below}')
print(f'Porcentaje sobre el total: {perc_below:.2f}%')

# Mostramos una muestra de los productos que necesitan reposición
display(df[df['below_reorder_point'] == True][['item_id', 'stock_level', 'reorder_point', 'below_reorder_point']].head())

Total de SKUs debajo del punto de reposición: 240
Porcentaje sobre el total: 7.49%


,item_id,stock_level,reorder_point,below_reorder_point
4,ITM10004,49,55,True
6,ITM10006,86,97,True
7,ITM10007,84,89,True
9,ITM10009,44,98,True
10,ITM10010,56,63,True


## Análisis de Riesgo de Stockout
Identificamos artículos donde el stock actual no cubrirá la demanda esperada durante el tiempo de entrega (`lead_time_days`).

In [17]:
# 1. Crear columna de riesgo
df['stockout_risk'] = df['stock_level'] < df['expected_demand_during_lead_time']

# 2. Cálculos generales
total_risk = df['stockout_risk'].sum()
perc_risk = (total_risk / len(df)) * 100

# 3. Intersección de condiciones
risk_and_below_reorder = df[df['stockout_risk'] & df['below_reorder_point']].shape[0]
risk_but_not_reorder = df[df['stockout_risk'] & ~df['below_reorder_point']].shape[0]

print(f"Total SKUs con riesgo de stockout: {total_risk}")
print(f"Porcentaje con riesgo: {perc_risk:.2f}%")
print(f"SKUs con riesgo Y debajo del punto de reposición: {risk_and_below_reorder}")
print(f"SKUs con riesgo PERO que aún no llegan al reorder point: {risk_but_not_reorder}")

# Visualizar una muestra de los casos críticos (riesgo sin reorder point activado)
display(df[df['stockout_risk'] & ~df['below_reorder_point']][['item_id', 'stock_level', 'expected_demand_during_lead_time', 'reorder_point']].head())

Total SKUs con riesgo de stockout: 818
Porcentaje con riesgo: 25.53%
SKUs con riesgo Y debajo del punto de reposición: 198
SKUs con riesgo PERO que aún no llegan al reorder point: 620


,item_id,stock_level,expected_demand_during_lead_time,reorder_point
2,ITM10002,132,301.52,60
5,ITM10005,154,215.70,62
14,ITM10014,160,164.88,58
16,ITM10016,155,193.62,20
19,ITM10019,147,223.14,54


## Valorización del Inventario
Calculamos el valor monetario del stock actual (`inventory_value = stock_level * unit_price`) y analizamos la distribución de este capital.

In [18]:
# 1. Calcular valor del inventario por SKU
df['inventory_value'] = df['stock_level'] * df['unit_price']

# 2. Métricas globales
valor_total = df['inventory_value'].sum()
valor_promedio = df['inventory_value'].mean()

print(f"Valor Total del Inventario: ${valor_total:,.2f}")
print(f"Valor Promedio por SKU: ${valor_promedio:,.2f}")

# 3. Top 10 SKUs por valor de inventario
top_10_valor = df[['item_id', 'category', 'stock_level', 'unit_price', 'inventory_value']].nlargest(10, 'inventory_value')

print("\nTop 10 SKUs con mayor valor almacenado:")
display(top_10_valor)

Valor Total del Inventario: $89,566,189.04
Valor Promedio por SKU: $27,954.49

Top 10 SKUs con mayor valor almacenado:


,item_id,category,stock_level,unit_price,inventory_value
2848,ITM12848,Pharma,491,199.90,98150.90
195,ITM10195,Groceries,494,197.52,97574.88
2117,ITM12117,Apparel,488,199.18,97199.84
775,ITM10775,Electronics,499,193.70,96656.30
2285,ITM12285,Pharma,479,199.97,95785.63
66,ITM10066,Apparel,493,194.29,95784.97
368,ITM10368,Pharma,490,194.64,95373.60
760,ITM10760,Groceries,475,198.76,94411.00
234,ITM10234,Electronics,481,194.32,93467.92
2154,ITM12154,Pharma,481,193.60,93121.60


## Análisis de Inventario por Categoría
Resumen agregado que muestra el stock físico, la valorización monetaria y la concentración de SKUs por cada categoría de producto.

In [19]:
# 1. Agrupar por categoría
cat_analysis = df.groupby('category').agg(
    total_stock=('stock_level', 'sum'),
    total_value=('inventory_value', 'sum'),
    sku_count=('item_id', 'count')
).reset_index()

# 2. Calcular porcentaje del valor total
valor_global = cat_analysis['total_value'].sum()
cat_analysis['perc_of_total_value'] = (cat_analysis['total_value'] / valor_global) * 100

# 3. Ordenar por valor total para mejor visualización
cat_analysis = cat_analysis.sort_values(by='total_value', ascending=False)

# Mostrar resultados con formato
print(f"Resumen por Categoría (Valor Total Global: ${valor_global:,.2f}):")
display(cat_analysis.style.format({
    'total_stock': '{:,}',
    'total_value': '${:,.2f}',
    'sku_count': '{:,}',
    'perc_of_total_value': '{:.2f}%'
}))

Resumen por Categoría (Valor Total Global: $89,566,189.04):


,category,total_stock,total_value,sku_count,perc_of_total_value
4,Pharma,"178,331","$19,142,268.69",660,21.37%
2,Electronics,"172,266","$18,234,365.99",651,20.36%
1,Automotive,"170,217","$18,042,426.13",658,20.14%
3,Groceries,"162,997","$17,342,651.88",618,19.36%
0,Apparel,"160,416","$16,804,476.35",617,18.76%


## Ratio de Cobertura vs Lead Time
Calculamos `coverage_to_lead_time_ratio = days_of_stock_coverage / lead_time_days`.
* Un ratio < 1 indica riesgo de quiebre antes de la reposición.
* Un ratio > 1 indica stock suficiente para cubrir el tiempo de espera.

In [23]:
# 1. Calcular el ratio
df['coverage_to_lead_time_ratio'] = df['days_of_stock_coverage'] / df['lead_time_days']

# 2. Calcular estadísticas descriptivas y percentiles específicos
ratio_stats = df['coverage_to_lead_time_ratio'].describe(percentiles=[.5, .75, .9])

# 3. Presentar resultados
print("Estadísticas del Ratio de Cobertura vs Lead Time:")
print(ratio_stats[['min', '50%', 'mean', '75%', '90%', 'max']])

# Identificar cuántos están por debajo del umbral crítico de 1
criticos = (df['coverage_to_lead_time_ratio'] < 1).sum()
print(f"\nSKUs con ratio < 1 (Riesgo estimado de stockout): {criticos} ({ (criticos/len(df))*100:.2f}%)")

Estadísticas del Ratio de Cobertura vs Lead Time:
min       0.061387
50%       2.046370
mean      4.640817
75%       4.638810
90%       9.751534
max     135.693215
Name: coverage_to_lead_time_ratio, dtype: float64

SKUs con ratio < 1 (Riesgo estimado de stockout): 818 (25.53%)


## Identificación de Candidatos a Sobrestock
Definimos candidatos a sobrestock basándonos en dos métricas:
1.  **Alta Cobertura:** Top 10% del `coverage_to_lead_time_ratio`.
2.  **Baja Rotación:** Bottom 25% del `turnover_ratio`.

In [21]:
# 1. Calcular los umbrales (cuantiles)
umbral_cobertura = df['coverage_to_lead_time_ratio'].quantile(0.90)
umbral_rotacion = df['turnover_ratio'].quantile(0.25)

# 2. Crear la columna booleana en el DataFrame original
df['overstock_candidate'] = (
    (df['coverage_to_lead_time_ratio'] >= umbral_cobertura) &
    (df['turnover_ratio'] <= umbral_rotacion)
)

# 3. Presentar resultados
print(f"Umbral de cobertura (Top 10%): {umbral_cobertura:.2f}")
print(f"Umbral de rotación (Bottom 25%): {umbral_rotacion:.2f}")
print(f"\nTotal de candidatos a sobrestock marcados: {df['overstock_candidate'].sum()}")

# Mostrar los principales candidatos utilizando la nueva columna
display(df[df['overstock_candidate']][['item_id', 'category', 'stock_level', 'coverage_to_lead_time_ratio', 'turnover_ratio', 'inventory_value']]
        .sort_values(by='inventory_value', ascending=False).head(10))

Umbral de cobertura (Top 10%): 9.75
Umbral de rotación (Bottom 25%): 4.59

Total de candidatos a sobrestock marcados: 70


,item_id,category,stock_level,coverage_to_lead_time_ratio,turnover_ratio,inventory_value
936,ITM10936,Electronics,471,18.847539,1.80,82985.49
2572,ITM12572,Electronics,409,11.345354,2.59,81767.28
1043,ITM11043,Electronics,398,25.644330,4.00,79249.76
949,ITM10949,Electronics,496,12.216749,3.25,79245.92
243,ITM10243,Groceries,416,14.444444,3.24,78203.84
685,ITM10685,Electronics,466,12.263158,4.43,72164.76
843,ITM10843,Electronics,388,15.967078,2.08,71869.24
2294,ITM12294,Groceries,453,12.681971,4.16,69879.78
2398,ITM12398,Automotive,476,12.962963,3.29,68829.60
2504,ITM12504,Automotive,400,9.955202,1.51,68704.00


In [24]:
df.to_csv("logistics_features.csv", index=False)

### Hallazgo principal: riesgo de stockout

Se identificaron **818 SKU (25,53%)** cuyo stock actual no sería suficiente para cubrir la demanda estimada durante su `lead_time`.

Dentro de este grupo, **198 SKU** ya se encuentran por debajo de su `reorder_point`, mientras que **620 SKU presentan riesgo de stockout aun estando por encima de su punto de reposición**.

Este resultado sugiere que, para una parte importante del inventario, los puntos de reposición podrían no estar reflejando adecuadamente la combinación entre demanda diaria y tiempo de reposición.

Estos casos deberían ser priorizados para una revisión de las políticas de reposición, especialmente en productos con alta demanda o lead times elevados.

## Feature Engineering Summary

Durante esta etapa se crearon nuevas variables orientadas a transformar los datos operativos originales en métricas útiles para la toma de decisiones.

Las principales variables generadas fueron:

- `days_of_stock_coverage`: cantidad estimada de días que el stock actual puede cubrir la demanda.
- `expected_demand_during_lead_time`: demanda esperada durante el tiempo necesario para recibir una reposición.
- `lead_time_stock_gap`: diferencia entre el stock disponible y la demanda estimada durante el lead time.
- `below_reorder_point`: indicador que identifica SKU que ya alcanzaron o superaron su punto de reposición.
- `stockout_risk`: indicador de productos cuyo stock actual podría no ser suficiente para cubrir la demanda durante el lead time.
- `inventory_value`: valor estimado del inventario almacenado por SKU.
- `coverage_to_lead_time_ratio`: relación entre la cobertura de stock disponible y el tiempo de reposición.
- `overstock_candidate`: indicador de SKU con cobertura relativa elevada y baja rotación, considerados posibles candidatos a sobrestock.

### Principales resultados

- **818 SKU (25,53%)** presentan riesgo estimado de stockout durante su lead time.
- De ellos, **620 SKU todavía se encuentran por encima de su punto de reposición**, lo que sugiere posibles oportunidades de revisión de los parámetros de reabastecimiento.
- El valor total estimado del inventario es de aproximadamente **89,57 millones de unidades monetarias**.
- Se identificaron **70 SKU candidatos a sobrestock**, definidos como productos pertenecientes al 10% con mayor cobertura relativa y al 25% con menor rotación.

Las variables desarrolladas en esta etapa serán utilizadas posteriormente para construir un indicador integral de riesgo de inventario y para profundizar el análisis operativo del depósito.